# Fine-tuning FLAN-T5 Base for mathematics with LoRA

I tried to teach **FLAN-T5 Base** to work through GSM8K word problems using LoRA. I trained for **five epochs** (an initial three-epoch run followed by two more epochs). Even after that, the answers I inspected were often wrong. This notebook documents the experiment and its limitations rather than presenting it as a successful mathematics tutor.

**Model:** `google/flan-t5-base`  
**Dataset:** `openai/gsm8k` (`main`)  
**Approach:** LoRA supervised fine-tuning of an encoder–decoder model

The notebook is a cleaned, reproducible version of my exploratory work. Training cells are intentionally left unexecuted; results reported in the notes come from the original experiment. The original notebook contains exploratory cells and commented-out continuation code, so the exact five-epoch run cannot be independently reconstructed from saved outputs alone.

## 1. Imports and setup

Install the packages in `requirements.txt` first. This experiment was developed with a CUDA-enabled PyTorch environment.

In [1]:
import re
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
)

MODEL_ID = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 2. Load and split GSM8K

I used 90% of the official training split for training, 10% for validation, and kept the official test split separate. I removed the `<<...>>` calculator annotations and changed `####` to `Final answer:`.

In [2]:
dataset = load_dataset("openai/gsm8k", "main")
splits = dataset["train"].train_test_split(test_size=0.1, seed=42)

def clean_example(example):
    answer = re.sub(r"<<.*?>>", "", example["answer"])
    return {"answer": answer.replace("####", "Final answer:").strip()}

train_clean = splits["train"].map(clean_example)
validation_clean = splits["test"].map(clean_example)
test_clean = dataset["test"].map(clean_example)
print(len(train_clean), len(validation_clean), len(test_clean))

Map:   0%|          | 0/6725 [00:00<?, ? examples/s]

Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

6725 748 1319


## 3. Tokenize inputs and target solutions

T5 is an encoder–decoder model. The **encoder** receives the question; the **decoder** learns to produce the worked solution. In my original experiment I used a maximum of 256 input tokens and 384 target tokens. Target padding is replaced by `-100` so it does not contribute to loss.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

START = "Solve the following mathematics problem step by step.\n\n"
END = "\n\nSolution: "

def tokenize_batch(batch):
    prompts = [START + question + END for question in batch["question"]]
    inputs = tokenizer(prompts, max_length=256, truncation=True,
                       padding="max_length")
    targets = tokenizer(text_target=batch["answer"], max_length=384,
                        truncation=True, padding="max_length")
    inputs["labels"] = [
        [token if token != tokenizer.pad_token_id else -100 for token in ids]
        for ids in targets["input_ids"]
    ]
    return inputs

train_tokens = train_clean.map(tokenize_batch, batched=True,
                               remove_columns=train_clean.column_names)
validation_tokens = validation_clean.map(tokenize_batch, batched=True,
                                         remove_columns=validation_clean.column_names)
print("Example input length:", len(train_tokens[0]["input_ids"]))

Map:   0%|          | 0/6725 [00:00<?, ? examples/s]

Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Example input length: 256


## 4. Add LoRA

I applied LoRA to T5's `q` and `v` projection layers, with rank 8 and alpha 16. The original notebook reported **884,736 trainable parameters** out of **248,462,592** (about **0.36%**).

In [4]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q", "v"], bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, label_pad_token_id=-100
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## 5. Train for three epochs

The original setup used a learning rate of `1e-4`, batch size 2, gradient accumulation 4, weight decay `0.01`, and epoch-level validation. This cell recreates that first stage.

In [5]:
first_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints/first_three_epochs",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=False,
    save_total_limit=2,
    report_to="none",
)
trainer = Seq2SeqTrainer(
    model=model, args=first_args,
    train_dataset=train_tokens, eval_dataset=validation_tokens,
    processing_class=tokenizer, data_collator=collator,
)
# Uncomment to run the first three epochs.
# first_result = trainer.train()

## 6. Continue for epochs four and five

After the initial three epochs, I continued training the same adapter for two more epochs. This is a **continuation**, not a new model trained from scratch. The source notebook kept this step as commented-out code; it did not preserve a complete executable training history.

**Important:** Run the previous `trainer.train()` first before executing this continuation. Otherwise the model will not have completed the first three epochs.

In [6]:
continued_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints/continued_two_epochs",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=False,
    save_total_limit=2,
    report_to="none",
)
continued_trainer = Seq2SeqTrainer(
    model=trainer.model, args=continued_args,
    train_dataset=train_tokens, eval_dataset=validation_tokens,
    processing_class=tokenizer, data_collator=collator,
)
# Run ONLY after the initial three-epoch stage has finished.
# continued_result = continued_trainer.train()
# continued_trainer.save_model("./adapters/flan_t5_math_lora_5_epochs")
# tokenizer.save_pretrained("./adapters/flan_t5_math_lora_5_epochs")

### Recorded loss values

The original notebook included the following values beside the commented-out continuation step:

| Recorded continuation epoch | Training loss | Validation loss |
|---|---:|---:|
| 1 | 5.611812 | 1.123578 |
| 2 | 5.541370 | 1.120934 |

These values are copied from the original notebook's notes. They should **not** be interpreted as a complete five-epoch loss history or as independently verified outputs from this cleaned notebook.

## 7. Test generated answers

Loss is useful, but the real question is whether the model gets the mathematics right. I used a small set of simple questions to inspect the generated solutions. This is a **qualitative check**, not a statistically meaningful GSM8K accuracy estimate.

In [7]:
def solve(question, trained_model):
    prompt = START + question + END
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=256).to(next(trained_model.parameters()).device)
    trained_model.eval()
    with torch.inference_mode():
        outputs = trained_model.generate(
            **inputs, max_new_tokens=300, do_sample=False
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

questions = [
    "Solve 3x + 7 = 22",
    "Solve 2x + 5 = 15",
    "What is 25% of 200?",
    "Simplify 3/4 + 1/4",
    "Solve x - 9 = 4",
]

# After training, inspect the final adapter's answers:
# for question in questions:
#     print("Question:", question)
#     print("Model:", solve(question, continued_trainer.model))
#     print("-" * 50)

### Optional: evaluate a small GSM8K sample

This checks **exact numerical final answers** on 20 official test questions. If the model never writes `Final answer:`, the extractor returns `None`. That means the score measures both correct arithmetic **and** adherence to the requested output format.

In [8]:
def extract_answer(text):
    matches = re.findall(
        r"Final answer:\s*\$?\s*(-?[\d,]+(?:\.\d+)?)",
        text, flags=re.IGNORECASE,
    )
    return float(matches[-1].replace(",", "")) if matches else None

def evaluate_sample(trained_model, examples=test_clean, count=20):
    correct = 0
    for i in range(min(count, len(examples))):
        example = examples[i]
        prediction = extract_answer(solve(example["question"], trained_model))
        expected = extract_answer(example["answer"])
        match = (prediction is not None and expected is not None
                 and abs(prediction - expected) < 1e-6)
        correct += int(match)
        print(f"{i+1:2}: predicted={prediction}, expected={expected}, correct={match}")
    print(f"Correct: {correct}/{min(count, len(examples))}")

# evaluate_sample(continued_trainer.model)

## 8. What I learned

I completed five epochs but still saw incorrect answers in my tests. The model sometimes produced numbers or repetitive text that did not solve the problem. I don't have a verified full-test-set accuracy result, so I won't claim that it scored 0% on GSM8K.

This experiment helped me understand the difference between **learning to imitate worked solutions** and **actually solving unseen problems**. It also gave me practice with encoder–decoder tokenization, label masking, LoRA, Hugging Face's `Seq2SeqTrainer`, and evaluating generated text.

For future work, I would use a larger, consistent held-out evaluation and compare against the unfine-tuned FLAN-T5 baseline before changing the training setup.